# 04 · Cache oluşturma (Kaggle, CPU)

**Bu notebook ne yapar?** 4.407 eğitim vakasının tüm serilerini **bir kez** okuyup
kanonik hacimlere çevirir ve `/kaggle/working/cache/<StudyInstanceUID>.npz` olarak yazar.
Eğitim notebook'u (05) bu çıktıyı girdi olarak bağlar ve DICOM'a hiç dokunmaz.

**Neden?** Bir vaka ~150 DICOM dosyası. Bunu her epoch'ta okumak eğitimi diske bağımlı
yapar (eski pipeline'ın asıl yavaşlığı buydu). Ön işleme ayrıntıları: `03_dicom_onisleme.ipynb`.

| ayar | değer | anlamı |
|---|---|---|
| `n_slices` | 16 | slot başına, yığının ortadaki %80'inden eşit aralıklı kesit |
| `size` | 224 | piksel |
| `fov_mm` | 160 | görüş alanı → 0.71 mm/piksel |
| slotlar | 6 | sagital/koronal/aksiyal × sıvı-duyarlı/T1 |

Tahmini çıktı: ~12 GB (sıkıştırılmış npz), süre ~1 saat (4 CPU).

**Kaggle ayarları:** Accelerator = None (CPU), Internet = kapalı olabilir,
Input = yarışma verisi + `rsna-knee-src` dataset'i.

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

## 0. Sıkıştırılmış DICOM çözücüleri

Veri setinde JPEG Lossless ve JPEG 2000 ile sıkıştırılmış seriler var. pydicom bunları ek
paketler olmadan açamaz; Kaggle imajında bu paketler yok. Kurulmazsa o seriler sessizce
atlanır (ilk cache denemesinde yaşadığımız sorun). Wheel'ler `rsna-knee-src` içinde
(`third_party/wheels`), internet gerekmez.

In [ ]:
from pathlib import Path
from src.config import IS_KAGGLE
from src.dicom import ensure_decoders

from src.config import find_input
WHEELS = [p for p in [find_input("third_party/wheels")] if p] if IS_KAGGLE else []
DECODERS = ensure_decoders(WHEELS)
print("DICOM çözücüler:", DECODERS)
if IS_KAGGLE:
    assert DECODERS["libjpeg"] and DECODERS["openjpeg"], "JPEG çözücüleri kurulamadı!"

In [ ]:
import time
import numpy as np
import pandas as pd

from src.config import DATA_DIR, EXPERIMENTS_DIR, IS_KAGGLE
from src.cache import CacheConfig, build_cache

SPLIT = "train"
N_WORKERS = 4
CFG = CacheConfig(n_slices=16, size=224, fov_mm=160.0)
OUT_DIR = EXPERIMENTS_DIR / "cache"

series = pd.read_csv(DATA_DIR / f"{SPLIT}_series.csv")
STUDIES = None  # None = hepsi
if not IS_KAGGLE:
    # Yerel deneme: yalnızca diskte DICOM'u olan test vakaları
    SPLIT = "test"
    series = pd.read_csv(DATA_DIR / "test_series.csv")
    STUDIES = [s for s in series.StudyInstanceUID.unique() if (DATA_DIR / "test_series" / s).exists()]
    N_WORKERS = 1
print(f"Veri: {DATA_DIR} | bölüm: {SPLIT} | vaka: {series.StudyInstanceUID.nunique() if STUDIES is None else len(STUDIES)}")

## 1. Cache'i oluştur
Yarıda kesilirse tekrar çalıştırmak yeterli: var olan `.npz` dosyaları atlanır.

In [ ]:
t0 = time.time()
meta = build_cache(series, DATA_DIR, SPLIT, OUT_DIR, CFG, n_workers=N_WORKERS, studies=STUDIES)
print(f"Bitti: {len(meta)} vaka, {(time.time() - t0) / 60:.1f} dk")

## 2. Kalite kontrol

Aşağıdaki tablolar eğitimden önce mutlaka okunmalı:
* **slot sayısı** dağılımı — 0 slotlu vaka = hiç okunamamış vaka.
* **hatalar** — bozuk seri/dosya sayısı.
* **taraf (laterality)** — etiketten mi geometriden mi geldi; ikisi birlikte varsa uyum oranı
  geometrik yöntemin güvenilirliğini gösterir.
* **düzlem uyuşmazlığı** — CSV'deki düzlem ile geometriden çıkan düzlem farklıysa.

In [ ]:
meta = pd.read_csv(OUT_DIR / "meta.csv")
print("slot sayısı dağılımı:\n", meta.n_slots.value_counts().sort_index().to_string())
print("\nhatası olan vaka:", meta.errors.fillna("").astype(bool).sum())
print("\ntaraf kaynağı:\n", meta.side_source.value_counts(dropna=False).to_string())
print("\ntaraf:\n", meta.side.value_counts(dropna=False).to_string())
both = meta.dropna(subset=["side_tag", "side_geometry"])
if len(both):
    print(f"\netiket ile geometri uyumu: {(both.side_tag == both.side_geometry).mean():.3f} (n={len(both)})")
print("\ndüzlem uyuşmazlığı olan vaka:", meta.plane_mismatch.fillna("").astype(bool).sum())

In [ ]:
import os
size_gb = sum(f.stat().st_size for f in OUT_DIR.glob("*.npz")) / 1e9
print(f"Cache boyutu: {size_gb:.2f} GB, dosya: {len(list(OUT_DIR.glob('*.npz')))}")
errs = meta[meta.errors.fillna("").astype(bool)]
errs.head(20)

**Sonraki adım:** Bu notebook'un çıktısını `05_egitim_kaggle.ipynb`'e Input olarak ekleyin.